Najnovejši hit je programabilni sesalec (model PS132). Vanj lahko vstavimo SD kartico z datoteko, ki je lahko videti, na primer, tako:

```
>10
^5
V2
v1
E
<3
>5
V0
>8
E
V2
E
E
<1
```

- `>10` pomeni, da se sesalec premakne za 10 polj desno ... in tako naprej.
- Poleg znaka `v` sesalec sprejme tudi `V` (veliko črko), ki (najbrž?) pomeni premik navzdol, čisto tako kot `v`.
- Znak `E` pomeni, da se sesalec ustavi. Če sesalec ne naleti na `E`, pač izvede datoteko do konca.

**V funkcijah, ki jih boste pisali, berite datoteke z zanko. Ne poskušajte jih prebrati v seznam (npr. `list(open("pot.txt"))`), saj je lahko datoteka tudi zeloooooo (točneje: neskončno) dolga.**

Pri reševanju naloge vam utegneta priti prav dve metodi nizov: [`split()`](https://docs.python.org/3/library/stdtypes.html#str.split) in [`strip()`](https://docs.python.org/3/library/stdtypes.html#str.strip). 

Še en koristen namig: kaj izpiše program

```python
f = open("planeti.txt")
for vrstica in f:
    print(vrstica)
    f.readline()
```

Poskusite. Pri reševanju boste potrebovali nekaj podobnega, samo ... malo več.

### 1. `izvedi`

Napiši funkcijo `izvedi(ime_datoteke)`, ki prejme ime datoteke in vrne končni koordinati sesalca.

#### Rešitev

Recimo tako.

In [5]:
smeri = {"<": (-1, 0), ">": (1, 0), "v": (0, -1), "V": (0, -1), "^": (0, 1)}

def izvedi(ime_datoteke):
    x = y = 0
    for vrstica in open(ime_datoteke):
        smer = vrstica[0]
        if smer == "E":
            break
        razdalja = int(vrstica[1:])
        dx, dy = smeri[smer]
        x += dx * razdalja
        y += dy * razdalja
    return x, y

Nič takšnega, česar še ne bi videli v prejšnjih nalogah.

Slovar `smeri`, s katerim si kot navadno pomagamo, smo dali ven iz funkcije, ker nam bo prišel prav tudi v naslednji funkciji.

### 2. `izvedi2`

Izkazalo se je, da smo spregledali nekaj v dokumentaciji: znak `V` ne pomeni pomika navzdol temveč preskok določenega števila vrstic datoteke. Tako `V2` pomeni, da preskočimo naslednji dve vrstici. Če imamo program
    
```python
>5
V2
<1  
E
>4
```

bo sesalec preskočil dve vrstici, namreč `<1` in `E`, torej naredi tole:

```python
>5  gre desno za 5, na (5, 0)
V2  preskoči naslednji dve vrstici, torej <1 in E
<1  to vrstico ignorira
E   tudi to ignorira
>4  gre desno za 4, na (9, 0)
```

Poleg tega se je izkazalo, da sesalec podpira komentarje: v vsaki vrstici lahko ukazu sledi presledek in nato poljubno besedilo, ki ga sesalec ignorora. Gornji zapis je torej veljaven program.

Tule je še ena zanimivost: program

```python
>5
V20
<2
```

bo deloval, saj sesalec preskoči naslednjih 20 vrstic, čeprav jih v datoteki ni toliko. Sesalec torej konča pot na (5, 0).

Napišite funkcijo `izvedi2(ime_datoteke)`, ki izvaja datoteke v tem formatu, torej:

- `Vnnn` pomeni, da program preskoči naslednjih `nnn` vrstic (kjer je `nnn` poljubno število).
- Ukazom lahko sledijo komentarji, ki se začnejo s presledkom in jih sesalec ignorira.

#### Rešitev

Naloga je v bistvu preverjala, ali vemo, kako se obnašajo datoteke: če znotraj zanke, ki bere datoteko, preberemo kakšno vrstico, zanka ne bo ponovno prebrala te vrstice.

In [3]:
def izvedi2(ime_datoteke):
    x = y = 0
    f = open(ime_datoteke)
    for vrstica in f:
        smer = vrstica[0]
        if smer == "E":
            break
        razdalja = int(vrstica.split()[0][1:])
        if smer == "V":
            for _ in range(razdalja):
                f.readline()
        else:
            dx, dy = smeri[smer]
            x += dx * razdalja
            y += dy * razdalja
    return x, y

Ko naletimo na `V`, iz datoteka `f` (z novo zanko) preberemo toliko vrstic, kolikor jih je treba preskočiti.

Druga rešitev (ki jo je napisal tudi vsaj en študent), je

In [4]:
def izvedi2(ime_datoteke):
    x = y = 0
    f = open(ime_datoteke)
    preskoci = 0
    for vrstica in f:
        if preskoci > 0:
            preskoci -= 1
            continue
        smer = vrstica[0]
        if smer == "E":
            break
        razdalja = int(vrstica.split()[0][1:])
        if smer == "V":
            preskoci = razdalja
        else:
            dx, dy = smeri[smer]
            x += dx * razdalja
            y += dy * razdalja
    return x, y

Tu spremenljivka `preskoci` pove, koliko vrstic je (še) treba preskočiti. Po branju vrstice preverimo, ali je `preskoci` različna od `0`; če je tako, zmanjšamo `preskoci` za `1` in nadaljujemo z naslednjo vrstico. `V` pa nastavi `preskoci`, kakor je treba.

### 3. `ocisti`

Če se vam zdi, da je to preskakovanje nesmiselno, prav tako pa je nesmiselno, da se sesalec ustavi ob nekem E, namesto da bi tu preprosto končali datoteko, imate prav.

Zato napišite funkcijo `ocisti(vhodna_datoteka, izhodna_datoteka)`, ki prejme ime vhodne datoteke in v izhodnno datoteko zapiše samo vrstice (vključno z morebitnimi komentarji), ki jih sesalec dejansko izvede. Vrstice, ki jih sesalec preskoči, ne zapiše, prav tako na piše vrstic, ki sledijo po tem, ko se sesalec že ustavi.

Če je vhodna datoteka

```
>5
V2  Preskoči naslednji dve vrstici
V2
E
<3  Gre levo
E   Konča
>100
```

potem bo izhodna datoteka vsebovala samo

```
>5
<3  Gre levo
```

#### Rešitev

Tole je pravzaprav enako (samo malo preprosteje) kot prejšnja naloga: namesto da premaknemo sesalec, zapišemo vrstico.

In [6]:
def ocisti(ime_datoteke, izhodna_datoteka):
    vhod = open(ime_datoteke)
    izhod = open(izhodna_datoteka, "w")
    for vrstica in vhod:
        if vrstica[0] == "V":
            for _ in range(int(vrstica.split()[0][1:].strip() or "0")):
                vhod.readline()
        elif vrstica[0] == "E":
            break
        else:
            izhod.write(vrstica)

Datoteki se "zapreta" sami, ko se funkcija konča. Če bi hoteli narediti po najnovejših pravilih, pa bi pisali

In [7]:
def ocisti(ime_datoteke, izhodna_datoteka):
    with open(ime_datoteke) as vhod, open(izhodna_datoteka, "w") as izhod:
        for vrstica in vhod:
            if vrstica[0] == "V":
                for _ in range(int(vrstica.split()[0][1:].strip() or "0")):
                    vhod.readline()
            elif vrstica[0] == "E":
                break
            else:
                izhod.write(vrstica)


Ali, malo lepše, z zlomljeno vrstico `with`, tako da vsako datoteko odpremo v svoji vrstici programa:

In [8]:
def ocisti(ime_datoteke, izhodna_datoteka):
    with open(ime_datoteke) as vhod, \
            open(izhodna_datoteka, "w") as izhod:
        for vrstica in vhod:
            if vrstica[0] == "V":
                for _ in range(int(vrstica.split()[0][1:].strip() or "0")):
                    vhod.readline()
            elif vrstica[0] == "E":
                break
            else:
                izhod.write(vrstica)
